# PDF of Sentinel data - Kraków vs Puszcza


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import rasterio

DATA = Path("./scenes")
S2_BANDS = ["B02", "B03", "B04"]
S1_BANDS = ["vv", "vh"]

def load_s2(prefix):
    out = {}
    for b in S2_BANDS:
        with rasterio.open(DATA / f"{prefix}_{b}.tif") as src:
            out[b] = src.read(1).astype(np.float32) / 10000.0
    return out

def load_s1(prefix):
    out = {}
    for b in S1_BANDS:
        with rasterio.open(DATA / f"{prefix}_{b}.tif") as src:
            x = src.read(1).astype(np.float32)
        out[b] = 10 * np.log10(np.maximum(x, 1e-10))
    return out

urban_s2, natural_s2 = load_s2("s2_urban"), load_s2("s2_natural")
urban, natural = load_s1("s1_urban"), load_s1("s1_natural")


## Sentinel-2


In [ ]:
for name, data in [("Puszcza", natural_s2), ("Kraków", urban_s2)]:
    plt.figure()
    for b in S2_BANDS:
        plt.hist(data[b].ravel(), bins=100, range=(0, 0.4), alpha=0.5, label=b, density=True)
    plt.xlabel("Reflectance"); plt.legend(); plt.title(name)
    plt.show()


**Observations.** Kraków: wider PDFs (mixed urban surfaces). Puszcza: narrower, darker visible bands (homogeneous forest canopy).


## Sentinel-1


In [ ]:
_, ax = plt.subplots(2, 2, figsize=(12, 12))
ax[0, 0].imshow(natural["vv"], cmap="gray", vmin=-20, vmax=0); ax[0, 0].set_title("VV Puszcza")
ax[0, 1].imshow(urban["vv"], cmap="gray", vmin=-20, vmax=0); ax[0, 1].set_title("VV Kraków")
ax[1, 0].imshow(natural["vh"], cmap="gray", vmin=-20, vmax=0); ax[1, 0].set_title("VH Puszcza")
ax[1, 1].imshow(urban["vh"], cmap="gray", vmin=-20, vmax=0); ax[1, 1].set_title("VH Kraków")
plt.show()

for name, data in [("Puszcza", natural), ("Kraków", urban)]:
    plt.figure()
    for b, c in [("vv", "C0"), ("vh", "C1")]:
        plt.hist(data[b].ravel(), bins=100, range=(-25, 15), alpha=0.5, label=b.upper(), color=c, density=True)
    plt.ylim(0, 0.2)
    plt.xlabel("Backscatter [dB]"); plt.legend(); plt.title(name)
    plt.show()


**Observations.** Kraków: broader / brighter VV (buildings, double bounce). Puszcza: lower, more compact backscatter (volume scattering).


## Steganography - replace 2 LSBs


In [ ]:
def replace_two_lsbs(image_float: np.ndarray, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    img8 = (np.clip(image_float, 0, 1) * 255).astype(np.uint8)
    cleared = img8 & 0b11111100
    noise = rng.integers(0, 4, size=img8.shape, dtype=np.uint8)
    manipulated = cleared | noise
    return manipulated.astype(np.float32) / 255.0

original = np.clip(natural_s2["B04"] / 0.4, 0, 1) 
manipulated = replace_two_lsbs(original)

_, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(original, cmap="gray"); ax[0].set_title("Puszcza - Original"); ax[0].axis("off")
ax[1].imshow(manipulated, cmap="gray"); ax[1].set_title("Puszcza - Manipulated"); ax[1].axis("off")
plt.show()

plt.figure()
plt.hist(original.ravel(), bins=100, range=(0, 1), alpha=0.5, label="original", density=True)
plt.hist(manipulated.ravel(), bins=100, range=(0, 1), alpha=0.5, label="manipulated", density=True)
plt.xlabel("Pixel value"); plt.legend(); plt.title("PDF - original vs 2-LSB replaced")
plt.show()


**Observations.** Visually almost no change (2 LSBs ≈ noise below perception). The PDF can show small differences (binning / discrete levels), but they are weak - alone this is a poor detector of satellite-image manipulation; you'd need LSB-focused / statistical steganalysis, not a plain histogram.
